In [ ]:
# === Setup ===
# Runtime: <1 minute fast, <2 minutes full on a typical CPU (estimate).
# Hardware: CPU ok; no GPU required.
# Network: none; all datasets are generated locally.
# Competition-safe: general profile; check the actual contest package/data policy.
# Cẩm nang P08: NumPy, pandas, sklearn, Matplotlib, joblib; no package installation.
import os
import random
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
random.seed(42)
np.random.seed(42)
FAST = os.environ.get('OAI_FAST_MODE', '0') == '1'
rng = np.random.default_rng(42)
OUT = Path('outputs')
OUT.mkdir(exist_ok=True)


# Validation — Đo đúng khả năng tổng quát hóa

Reference thực hành. Dự đoán trước mỗi experiment, rồi ghi Result → Observation → Why.

## Data → EDA

Label được gán ngẫu nhiên cho mỗi người, các lần đo gần giống fingerprint người đó. Cơ chế tạo dữ liệu chủ đích loại tín hiệu tổng quát; mô hình chỉ có thể nhớ người đã gặp.

In [ ]:
from sklearn.model_selection import (GroupKFold, GroupShuffleSplit,
    StratifiedKFold, KFold, TimeSeriesSplit)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import f1_score

group_count = 120 if FAST else 240
groups = np.repeat(np.arange(group_count), 4)
fingerprints = rng.normal(size=(group_count, 10))
group_labels = rng.integers(0, 2, size=group_count)
X = fingerprints[groups] + rng.normal(0, 0.01, (len(groups), 10))
y = group_labels[groups]
test_groups = np.repeat(np.arange(32), 4)
X_test = rng.normal(size=(32, 10))[test_groups] + rng.normal(0, 0.01, (128, 10))
test_ids = np.array([f'test_{i}' for i in range(len(X_test))])

def build_model():
    """Return an unfitted nearest-neighbor pipeline for X (n, 10)."""
    return make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=1))

print('labeled shape:', X.shape)
print('class counts:', np.bincount(y))
assert np.isfinite(X).all()


## Preprocess → Model → Train

Scaler phải fit lại trong mỗi fold. Dự đoán row split và group split sẽ cho kết luận khác nhau.

In [ ]:
def run_cv(splitter, enforce_groups):
    """Return OOF (n, 2), fold table, assignment (n,) with exactly-once coverage."""
    oof = np.full((len(y), 2), np.nan)
    coverage = np.zeros(len(y), dtype=int)
    assignment = np.full(len(y), -1)
    rows = []
    splits = splitter.split(X, y, groups) if enforce_groups else splitter.split(X, y)
    for fold, (tr, va) in enumerate(splits):
        overlap = len(set(groups[tr]) & set(groups[va]))
        if enforce_groups:
            assert overlap == 0
        fitted = build_model().fit(X[tr], y[tr])
        assert fitted.classes_.tolist() == [0, 1]
        oof[va] = fitted.predict_proba(X[va])
        coverage[va] += 1
        assignment[va] = fold
        score = f1_score(y[va], oof[va].argmax(axis=1), labels=[0, 1],
                         average='macro', zero_division=0)
        rows.append({'fold': fold, 'macro_f1': score, 'group_overlap': overlap,
                     'validation_rows': len(va), 'positive_count': int(y[va].sum())})
    assert np.all(coverage == 1)
    assert np.isfinite(oof).all() and np.allclose(oof.sum(axis=1), 1)
    return oof, pd.DataFrame(rows), assignment

row_oof, row_table, _ = run_cv(StratifiedKFold(3, shuffle=True, random_state=42), False)
group_oof, group_table, assignment = run_cv(GroupKFold(3), True)
assert group_table['group_overlap'].sum() == 0
assert row_table['group_overlap'].sum() > 0
print('group folds:')
print(group_table.to_string(index=False))


## Evaluate — Mean, spread và OOF

**Hypothesis:** random-row CV lạc quan vì cùng fingerprint xuất hiện ở train và validation. **Result:** bảng sau. **Observation:** so overlap và điểm thực đo. **Why:** label group ngẫu nhiên không chứa quy luật cho người mới.

In [ ]:
comparison = []
for label, probs, table in [('row_stratified', row_oof, row_table),
                             ('group', group_oof, group_table)]:
    comparison.append({'split': label, 'mean': table['macro_f1'].mean(),
                       'sample_std': table['macro_f1'].std(ddof=1),
                       'oof_f1': f1_score(y, probs.argmax(axis=1), labels=[0, 1],
                                          average='macro', zero_division=0),
                       'group_overlap_sum': table['group_overlap'].sum()})
print(pd.DataFrame(comparison).to_string(index=False))
pd.DataFrame({'row': np.arange(len(y)), 'group': groups, 'fold': assignment,
              'p0': group_oof[:, 0], 'p1': group_oof[:, 1]}).to_csv(
                  OUT / 'group_oof_solution.csv', index=False)


### Transfer — KFold và thời gian

KFold chỉ hợp lý cho dòng độc lập. TimeSeriesSplit cần thứ tự thời gian có nghĩa; ví dụ dưới có một bước gap. Dòng prefix không có OOF, nên không assert coverage=1 cho toàn chuỗi.

In [ ]:
independent_rows = np.arange(12)
kfold_sizes = [(len(tr), len(va)) for tr, va in KFold(3).split(independent_rows)]
assert kfold_sizes == [(8, 4)] * 3
timeline = np.arange(20)
time_windows = []
for tr, va in TimeSeriesSplit(n_splits=3, test_size=4, gap=1).split(timeline):
    assert timeline[tr].max() < timeline[va].min()
    assert va.min() - tr.max() == 2
    time_windows.append({'train_end': int(tr.max()), 'valid_start': int(va.min()),
                         'valid_end': int(va.max())})
print(pd.DataFrame(time_windows).to_string(index=False))


## Submit

Giữ mục tiêu người mới. Refit baseline trên labeled data sau khi đánh giá; public test không có label.

In [ ]:
model = build_model().fit(X, y)
test_pred = model.predict(X_test)


In [ ]:
# WHY: validate the file read from disk, not only the in-memory frame.
submission = pd.DataFrame({'id': test_ids, 'label': test_pred})
assert submission.columns.tolist() == ['id', 'label']
assert len(submission) == len(test_ids)
assert submission['id'].is_unique
assert submission['label'].isin([0, 1]).all()
submission.to_csv(OUT / 'submission_solution.csv', index=False)
reloaded = pd.read_csv(OUT / 'submission_solution.csv')
assert reloaded['id'].tolist() == list(test_ids)
assert reloaded['label'].tolist() == list(test_pred)
print('submission rows:', len(reloaded))


## Postmortem

Ghi lại đơn vị dự đoán, group overlap, coverage và giới hạn generalization. Không gọi điểm CV thấp là lỗi code khi dữ liệu không có tín hiệu cho người mới.